# Twin Engine — Phase 1 : Détection YOLO-seg + Projection 3D

Notebook Colab pour tester la perception industrielle sans installer le venv local.

**Pipeline** : image RGB → YOLO-seg → masques → position 3D (mono) → `twin_map.json`

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy pyyaml matplotlib tqdm

## 1. Récupérer le code Twin Engine

Option A : cloner le repo (remplacer l'URL par la vôtre).

Option B : uploader un zip du dossier `twin-engine/` via le panneau fichiers Colab.

In [ ]:
import sys
from pathlib import Path

# --- Option A : git clone (décommenter et adapter) ---
# !git clone https://github.com/VOTRE_ORG/twin-engine.git
# REPO_ROOT = Path('/content/twin-engine')

# --- Option B : repo déjà présent / uploadé ---
REPO_ROOT = Path('/content/twin-engine')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('.').resolve()

sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT =', REPO_ROOT)
assert (REPO_ROOT / 'semantic' / 'detector.py').exists(), 'twin-engine non trouvé — clone ou upload requis'

## 2. Charger une image

Uploader une photo d'atelier/usine, ou utiliser une image de démo.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()  # sélectionner une image JPG/PNG
image_path = next(iter(uploaded.keys()))
bgr = cv2.imread(image_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.axis('off')
plt.title('Image source')
plt.show()

## 3. Détection YOLO-seg industrielle

In [ ]:
from semantic.detector import IndustrialDetector

CLASSES_CONFIG = REPO_ROOT / 'config' / 'semantic' / 'classes.yaml'
detector = IndustrialDetector(CLASSES_CONFIG)
detections = detector.detect_frame(rgb, timestamp=0.0)

print(f'Détections : {len(detections)}')
for d in detections:
    print(f'  {d.class_name} ({d.coco_label}) conf={d.confidence:.2f}')

In [ ]:
colors = detector.get_class_colors(CLASSES_CONFIG)
overlay = rgb.copy()

for det in detections:
    color = colors.get(det.class_name, [1.0, 0.0, 0.0])
    color_u8 = [int(c * 255) for c in color]
    x1, y1, x2, y2 = map(int, det.bbox_xyxy)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), color_u8, 2)
    if det.mask is not None:
        overlay[det.mask] = (
            0.5 * overlay[det.mask] + 0.5 * np.array(color_u8)
        ).astype(np.uint8)
    cv2.putText(
        overlay, det.class_name, (x1, max(y1 - 5, 0)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_u8, 1,
    )

plt.figure(figsize=(10, 6))
plt.imshow(overlay)
plt.axis('off')
plt.title('Détections industrielles (proxy COCO)')
plt.show()

## 4. Projection 3D (mode mono, pose identité)

Sans depth map ni SLAM : profondeur estimée via `(fy × hauteur_objet) / hauteur_bbox`.

In [ ]:
from semantic.projector import CameraPose, ObjectProjector
from mapping.fusion import fuse_from_config

SLAM_CONFIG = REPO_ROOT / 'config' / 'slam' / 'tum_rgbd.yaml'
projector = ObjectProjector.from_slam_config(SLAM_CONFIG, CLASSES_CONFIG)
pose = CameraPose.identity(0.0)

observations = []
for det in detections:
    obj = projector.project_detection_mono(det, pose)
    if obj is not None:
        observations.append(obj)

twin_map = fuse_from_config(observations, CLASSES_CONFIG, scene_name='colab_demo')
print(f'Objets 3D : {len(twin_map.objects)}')
for obj in twin_map.objects:
    p = obj.position
    print(f'  {obj.class_name}: ({p.x:.2f}, {p.y:.2f}, {p.z:.2f}) conf={obj.confidence:.2f}')

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, c='blue', marker='^', s=80, label='caméra')

for obj in twin_map.objects:
    c = colors.get(obj.class_name, [0.8, 0.8, 0.2])
    ax.scatter(
        obj.position.x, obj.position.y, obj.position.z,
        c=[c], s=60, label=obj.class_name,
    )

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Carte objet-centric (échelle relative)')
ax.legend(loc='upper left', fontsize=8)
plt.show()

## 5. Export `twin_map.json`

In [ ]:
out_path = REPO_ROOT / 'output' / 'twin_map.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
twin_map.save_json(out_path)
print(out_path.read_text(encoding='utf-8')[:2000])
files.download(str(out_path))

## 6. (Optionnel) Pipeline TUM RGB-D

Si vous montez le dataset TUM `freiburg1_xyz` sur Colab :

In [ ]:
# !wget -q -O tum_xyz.tgz https://vision.in.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_xyz.tgz
# !tar xzf tum_xyz.tgz

# !python scripts/run_pipeline.py \
#   --dataset tum \
#   --dataset-root rgbd_dataset_freiburg1_xyz \
#   --identity-poses \
#   --skip-cad-retrieval \
#   --no-view \
#   --stride 20 \
#   --max-frames 30 \
#   --scene-name colab_tum